In [1]:
import numpy as np
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer

print("=== DEPLOYING AI QUALITY ENGINEERING SUITE ===")

# 1. CONNECTIVITY SETUP
PINECONE_API_KEY = "<Enter your APIs>" # <-- Paste your secret token here
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index("infra-cost-index")

# Load our embedding engine locally on our CPU instance
model = SentenceTransformer('all-MiniLM-L6-v2')

=== DEPLOYING AI QUALITY ENGINEERING SUITE ===


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
# 2. DEFINING THE QUALITY ENVIRONMENT TEST GROUND TRUTH
# We want to test if our database can accurately surface cost metrics
test_query = "Give me the cost of our API Gateway proxy"
ground_truth_keyword = "$7.00" # We know the true source document contains this exact value

print(f"Test Query: '{test_query}'")
print(f"Expected Ground Truth Token: '{ground_truth_keyword}'\n")

Test Query: 'Give me the cost of our API Gateway proxy'
Expected Ground Truth Token: '$7.00'



In [3]:
# 3. EXECUTE THE RETRIEVAL PHASE (Requesting Top 3 Chunks)
query_vector = model.encode(test_query).tolist()

# top_k=3 fetches the top 3 matches to evaluate our Hit Rate window
raw_results = index.query(
    vector=query_vector,
    top_k=3,
    include_metadata=True
)

In [4]:
# 4. QUALITY ENGINEERING METRIC EVALUATION LOOPS
print("--- Analyzing Retrieved Payload ---")
chunks_retrieved = raw_results['matches']

hit_detected = False
total_character_count = 0
relevant_character_count = 0

for rank, match in enumerate(chunks_retrieved, start=1):
    text_content = match['metadata']['raw_text']
    text_length = len(text_content)
    total_character_count += text_length
    
    # Check if the ground truth answer is nested in this specific chunk
    contains_answer = ground_truth_keyword in text_content
    
    print(f" Rank [{rank}] | Vector Score: {match['score']:.4f} | Contains Answer: {contains_answer}")
    
    if contains_answer:
        hit_detected = True
        # Track the character volume of the winning chunk
        relevant_character_count += text_length

--- Analyzing Retrieved Payload ---
 Rank [1] | Vector Score: 0.2541 | Contains Answer: False


In [5]:
# 5. MATHEMATICAL FORMULATION OF AI METRICS
print("\n=== FINAL AUTOMATED QUALITY METRIC REPORT ===")

# Metric A: Hit Rate Calculations
hit_rate_score = 1.0 if hit_detected else 0.0
print(f"RETRIEVAL HIT RATE:            {hit_rate_score * 100:.1f}% " + (" [PASS]" if hit_detected else " [FAIL]"))

# Metric B: Noise Ratio Calculations
if total_character_count > 0:
    noise_ratio = (total_character_count - relevant_character_count) / total_character_count
else:
    noise_ratio = 1.0

print(f"TOTAL SYSTEM CONTEXT NOISE:    {noise_ratio * 100:.2f}%")
print(f" -> Total Data Transferred:    {total_character_count} characters")
print(f" -> Meaningful Data Volume:    {relevant_character_count} characters")
print("=============================================================")


=== FINAL AUTOMATED QUALITY METRIC REPORT ===
RETRIEVAL HIT RATE:            0.0%  [FAIL]
TOTAL SYSTEM CONTEXT NOISE:    100.00%
 -> Total Data Transferred:    79 characters
 -> Meaningful Data Volume:    0 characters
